# BCF Post-Processing

BCF Post-Processing — ATE, Posterior Projection, and CATE by Moderator (Dictator + Trust)

Tristan Muno [](https://orcid.org/0009-0002-3078-8436) (University of Mannheim)  
Vera Okisheva (University of Mannheim)  
Raphael Klöckner (University of Mannheim)  
June 11, 2026

## Purpose

This notebook post-processes the **robustness** estimator — the saved hierarchical BCF fits (`data/03_final/bcf_{dictator,trust}.rds`) — into the same three quantities, in the same order, as the combined view of the primary GRF notebook `07_postprocess_grf.qmd`:

1.  **Average treatment effect (ATE)** — the posterior of the ATE for each game.
2.  **Posterior linear projection** — the Bayesian counterpart to GRF’s best linear projection (`woody2021model`): each posterior draw of τ(x) is projected onto the moderator design, giving a posterior distribution of projection coefficients. Appendix-style linear summary.
3.  **CATE by moderator** — the headline figures: estimated CATE on the y-axis, a moderator on the x-axis.

It is the BCF counterpart to `07_postprocess_grf.qmd` and is structurally parallel to it; only the inference machinery differs. GRF reports point estimates with honest cluster-robust confidence intervals; BCF reports **posterior medians** (read off the τ(x) draws from `get_forest_fit()`) with 95% credible intervals for the ATE and projection, and a cluster-robust band for the CATE-by-moderator figures that adds a respondent cluster bootstrap to the posterior SD (see @sec-cluster-bootstrap).

Following the **combined-figure** grammar of `07` (its `# Combined figures` section) and the house style of `09_additional_figs_tables.qmd`, every figure overlays **both games on one canvas**. The two games are the only thing encoded by colour, shape, and linetype, and they are always encoded by all three together, so each game is distinguishable in colour, in greyscale print, and under colourblindness. Every other distinction (which moderator, which level) is carried by facets and axis position.

Two moderator families, matching the research plan:

- **Conjoint round-level** moderators (`prof_mods`) — other attributes in the same profile.
- **Individual-level** moderators (`resp_mods`) — respondent pre-treatment items and indices.

Per-level uncertainty is a **cluster-robust 95% interval**: the posterior median of the bin/level CATE (cf. `fig-cate-by-cov-bcf` in `03_multibart_nested_ri_test.qmd`) with a band that combines the posterior SD (model/estimation uncertainty) and a **respondent cluster bootstrap** SD (within-person dependence across the three rounds; see @sec-cluster-bootstrap). We added the bootstrap to test the worry that the bands looked too narrow at low-count levels: the answer is that within-respondent clustering does **not** explain it — the posterior SD dominates the bootstrap SD by roughly an order of magnitude at every level, and it already widens appropriately as the level count shrinks. The bands are, in other words, genuinely that precise; the bootstrap is folded in for completeness but moves them by only a few percent.

## Setup

The setup chunk loads the project packages plus `multibart`’s own dependencies before the local package is attached (needed for `get_forest_fit()`).

In [ ]:
#| label: setup
#| include: false
# execution time
start_time <- Sys.time()
# console width
options(width = 80)
# packages
p_required <- c(
  "tidyverse", # dplyr, ggplot2, tidyr, purrr
  "here", # relative paths
  "ggpubr", # theme_pubr
  "patchwork", # combined-figure composition
  "sessioninfo", # session docs

  # multibart dependencies -----------------------------------------------------
  "Rcpp", # multibart Depends + LinkingTo
  "RcppArmadillo", # multibart Depends + LinkingTo
  "Rcereal", # multibart Depends + LinkingTo
  "fastDummies" # multibart helper dependency
)
packages <- rownames(installed.packages())
p_to_install <- p_required[!(p_required %in% packages)]
if (length(p_to_install) > 0) {
  install.packages(p_to_install)
}
sapply(p_required, require, character.only = TRUE)

Loading required package: tidyverse

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: here

here() starts at C:/R/research/MBEU25

Loading required package: ggpubr

Loading required package: patchwork

Loading required package: sessioninfo

Loading required package: Rcpp

Loading required package: RcppArmadillo

Loading required package: Rcereal

Loading required package: fastDummies

    tidyverse          here        ggpubr     patchwork   sessioninfo 
         TRUE          TRUE          TRUE          TRUE          TRUE 
         Rcpp RcppArmadillo       Rcereal   fastDummies 
         TRUE          TRUE          TRUE          TRUE 

In [ ]:
#| label: setup-multibart
#| include: false
# install local source on first use
if (!requireNamespace("multibart", quietly = TRUE)) {
  install.packages(
    here::here("code", "multibart"),
    repos = NULL,
    type = "source"
  )
}
# attach installed package
library(multibart)

## Load data and fits

In [ ]:
#| label: load-data
#| cache: false
dat <- readRDS(here::here("data", "02_processed", "eu25_long.rds"))
fit_dict <- readRDS(here::here("data", "03_final", "bcf_dictator.rds"))
fit_trust <- readRDS(here::here("data", "03_final", "bcf_trust.rds"))

## Variable definitions

The frozen moderator list and `prep_game()` are reused verbatim from `05_bcf_fit.qmd` so the reconstructed moderator design matches fit-time row order and columns exactly.

In [ ]:
#| label: moderators
prof_mods <- c(
  "cj_nat",
  "der_cj_eu_identity",
  "der_cj_partisanship",
  "cj_age",
  "cj_sex",
  "cj_class"
)
resp_mods <- c(
  "q_gender",
  "q_age",
  "q_identity_country",
  "q_identity_eu",
  "q_identity_europe",
  "q_religion",
  "q_class",
  "q_eu_efficacy_understand",
  "q_pop_reps",
  "q_pop_goodevil",
  "q_pop_compromise",
  "q_dem_compromise",
  "q_dem_listen",
  "q_tech_experts",
  "q_tech_leaders",
  "q_party_harm",
  "q_people_incompetent",
  "q_eu_longterm",
  "q_eu_responsive",
  "q_eu_imp_nat_econ",
  "q_eu_imp_nat_cul",
  "q_eu_imp_nat_pol",
  "q_eu_abolish",
  "q_eu_satisfaction",
  "q_rural_urban",
  "q_edu_age_stop"
)

In [ ]:
#| label: prep-game
prep_game <- function(dat, game) {
  sub <- dat |>
    filter(cj_game_type == game) # full per-game sample (no subsampling)

  y <- sub$cj_pl2
  z <- as.integer(sub$cj_rel == "muslim") # Muslim vs all-else pooled
  country_id <- factor(sub$meta_country)
  respondent_id <- factor(sub$meta_pid)

  mod_df <- sub |>
    select(all_of(c(prof_mods, resp_mods))) |>
    mutate(across(where(is.character), factor))
  sub_X <- model.matrix(~ . - 1, data = mod_df) # factors -> dummies

  stopifnot(
    !anyNA(sub_X),
    all(z %in% c(0L, 1L)),
    !anyNA(y),
    nrow(sub_X) == length(y)
  )

  list(
    y = y,
    z = z,
    sub_X = sub_X,
    country_id = country_id,
    respondent_id = respondent_id
  )
}

`game_mods()` returns the original (pre-dummy) moderator values for one game, in the same row order as `prep_game()`, so each observation’s τ̂ aligns with its moderator value.

In [ ]:
#| label: game-mods
game_mods <- function(dat, game) {
  dat |>
    filter(cj_game_type == game) |>
    select(all_of(c(prof_mods, resp_mods))) |>
    mutate(across(where(is.character), factor))
}

## House style

The Okabe-Ito game palette with redundant shape and linetype, the three reserved scales, the shared theme, and the reference layers — copied from `09_additional_figs_tables.qmd` / the combined section of `07_postprocess_grf.qmd` so the BCF figures share the manuscript grammar exactly.

In [ ]:
#| label: game-style
game_labs <- c(cj_dict = "Dictator game", cj_trust = "Trust game")
game_cols <- c("Dictator game" = "#0072B2", "Trust game" = "#D55E00")
game_shapes <- c("Dictator game" = 16, "Trust game" = 17) # circle / triangle
game_ltys <- c("Dictator game" = "solid", "Trust game" = "longdash")

scale_colour_game <- function() {
  scale_colour_manual(values = game_cols, name = NULL)
}
scale_shape_game <- function() {
  scale_shape_manual(values = game_shapes, name = NULL)
}
scale_linetype_game <- function() {
  scale_linetype_manual(values = game_ltys, name = NULL)
}

# shared moderator labels + categorical-first ordering for the facets
source(here::here("code", "helper_scripts", "moderator_labels.R"))

theme_mbeu <- function(base_size = 11) {
  ggpubr::theme_pubr(base_size = base_size) +
    theme(legend.position = "top")
}
ref_zero <- geom_hline(yintercept = 0, linetype = "dashed", colour = "grey60")
dodge <- position_dodge(width = 0.5)
yl_cate <- expression(widehat(CATE))
yl_ric <- expression(hat(tau)[ric]) # profile-level CATE
yl_i <- expression(hat(tau)[ic]) # respondent-level CATE

## Posterior summary builders

Every quantity below is read off `tau_draws`, the `nsim × n` posterior matrix returned by `get_forest_fit()` (rows index posterior draws, columns index observations). All three builders operate on a single game’s `tau_draws`, so they run inside the per-game block and the large matrix is freed before the next game.

A moderator is **discrete** (factor, or numeric with ≤ 10 distinct values: Likert items) or **binned** (higher-cardinality numeric → 10 quantile bins).

In [ ]:
#| label: classify
classify_mod <- function(col) {
  if (is.factor(col) || is.character(col)) {
    return("discrete")
  }
  if (dplyr::n_distinct(col) <= 10) {
    return("discrete")
  }
  "binned"
}

Shared display specs, matching `07_postprocess_grf.qmd`: the substantive `q_class` order (`dont_know` last), the 5-point-scale reversal (`6 - x`) applied to the identity scales (so higher means more attachment) and to the “EU effect on” items (so higher means a more positive evaluation), the two fixed display grids for the genuinely continuous moderators, and the number of cluster bootstrap resamples.

In [ ]:
#| label: display-specs
class_order <- c(
  "working_class",
  "lower_middle_class",
  "middle_class",
  "upper_middle_class",
  "higher_class",
  "dont_know",
  "other"
)
flip5 <- \(x) 6 - x # 5-point scales: reverse so higher = more of the construct
cont_grids <- list(
  q_age = seq(18, 75, length.out = 101), # 100 intervals over the full range
  q_edu_age_stop = seq(14.5, 30.5, by = 1) # integer years 15..30
)
boot_B <- 500L # respondent cluster bootstrap resamples

`ate_summary()` returns the posterior of the ATE: the per-draw mean of τ(x) over all observations, summarised by its median and a 95% credible interval.

In [ ]:
#| label: ate-summary
ate_summary <- function(tau_draws) {
  bd <- rowMeans(tau_draws) # posterior of the ATE
  tibble(
    estimate = median(bd),
    lower = unname(quantile(bd, 0.025)),
    upper = unname(quantile(bd, 0.975))
  )
}

`proj_summary()` is the Bayesian posterior projection (`woody2021model`), the BCF counterpart to GRF’s `best_linear_projection`. For each posterior draw `d`, the coefficient vector is $\beta_d = (X'X)^{-1} X' \tau_d$; we stack these into a posterior and summarise each term by its median and 95% credible interval. The design `sub_X` is `model.matrix(~ . - 1, ...)`, which is full rank (only the first factor is fully expanded; the rest are contrast-coded), so `crossprod(sub_X)` inverts directly — no added intercept, no ridge. It is the same column basis as GRF’s `X_dict`, so the terms are comparable. The only large multiply (`tau_draws %*% sub_X`) returns a small `nsim × p` matrix, so the step is memory-safe.

In [ ]:
#| label: proj-summary
proj_summary <- function(tau_draws, sub_X) {
  xtx <- crossprod(sub_X) # p x p
  m <- tau_draws %*% sub_X # nsim x p  (= tau_d' X per draw)
  beta <- solve(xtx, t(m)) # p x nsim  = (X'X)^{-1} X' tau_d
  tibble(
    term = colnames(sub_X),
    estimate = apply(beta, 1, median),
    lower = apply(beta, 1, quantile, 0.025),
    upper = apply(beta, 1, quantile, 0.975)
  )
}

`level_keeps()` enumerates the kept levels/bins of one moderator and returns, for each, the logical keep-vector over the game’s rows together with its display `level`, `order`, and `is_scale` tag. It mirrors the combined builder in `07` exactly — discrete levels honor the factor order and are dropped below **1% of the game sample** (their CATE is too imprecise to compare); `cj_age` is treated as categorical (fixed conjoint ages); and the two genuinely continuous moderators are binned on the **fixed display grid** (`cont_grids`) rather than into deciles, so the line spans the full range. Sharing this enumeration between the posterior summary and the cluster bootstrap guarantees the two use identical bins.

In [ ]:
#| label: level-keeps
level_keeps <- function(mm, mod) {
  v <- mm[[mod]]
  is_num <- is.numeric(v)
  # genuinely continuous moderators -> fixed grid (or decile fallback)
  if (is_num && length(unique(v)) > 12) {
    if (mod %in% names(cont_grids)) {
      br <- cont_grids[[mod]]
      cell <- cut(v, breaks = br, include.lowest = TRUE)
      mids <- (head(br, -1) + tail(br, -1)) / 2
      out <- lapply(
        which(tabulate(cell, nbins = length(mids)) > 0),
        function(k) {
          keep <- !is.na(cell) & as.integer(cell) == k
          list(
            level = as.character(round(mids[k])),
            order = mids[k],
            is_scale = TRUE,
            keep = keep
          )
        }
      )
      return(out)
    }
    bin <- ntile(v, 10)
    return(lapply(sort(unique(bin)), function(k) {
      keep <- !is.na(bin) & bin == k
      list(
        level = as.character(round(mean(v[keep]))),
        order = mean(v[keep]),
        is_scale = TRUE,
        keep = keep
      )
    }))
  }
  # discrete moderators: honor factor order; drop levels below 1% of the game sample
  levs <- if (is_num) {
    sort(unique(v))
  } else if (is.factor(v)) {
    levels(droplevels(v))
  } else {
    sort(unique(as.character(v)))
  }
  out <- lapply(seq_along(levs), function(i) {
    lev <- levs[i]
    keep <- if (is_num) {
      !is.na(v) & v == lev
    } else {
      !is.na(v) & as.character(v) == lev
    }
    if (sum(keep) / sum(!is.na(v)) < 0.01) {
      return(NULL)
    }
    list(
      level = as.character(lev),
      order = if (is_num) as.numeric(lev) else i,
      is_scale = is_num && mod != "cj_age",
      keep = keep
    )
  })
  out[!vapply(out, is.null, logical(1))]
}

`cate_one_bcf()` reads each bin’s estimate off the **posterior of the bin mean** (`rowMeans` of τ(x) over the bin’s observations): the point is the posterior median and `post_sd` is the posterior standard deviation, the **model/estimation** uncertainty. The final interval (built downstream) adds the respondent-cluster bootstrap variance to this.

In [ ]:
#| label: cate-one-bcf
# posterior of the bin CATE over a logical keep-vector (length n)
.bin_post <- function(tau_draws, keep) {
  bd <- rowMeans(tau_draws[, keep, drop = FALSE]) # posterior of bin mean
  c(estimate = median(bd), post_sd = sd(bd))
}

cate_one_bcf <- function(tau_draws, mm, mod, game_label) {
  ks <- level_keeps(mm, mod)
  bind_rows(lapply(ks, function(L) {
    p <- .bin_post(tau_draws, L$keep)
    tibble(
      mod = mod,
      level = L$level,
      order = L$order,
      is_scale = L$is_scale,
      game = game_label,
      n = sum(L$keep),
      estimate = unname(p["estimate"]),
      post_sd = unname(p["post_sd"])
    )
  }))
}

## Respondent cluster bootstrap

The posterior `post_sd` above captures **model/estimation** uncertainty in the regularized τ(x) surface alone — not the within-respondent design dependence (each respondent contributes three conjoint rounds) and not the sampling variability of which respondents land in a bin. The worry was that omitting this clustering made the bands look too narrow, especially at sparsely populated levels. We test that directly with a **respondent cluster bootstrap**: resample respondents (the clusters) with replacement `boot_B` times, recompute each bin’s mean of the per-observation point estimate `tau_hat`, and take the bootstrap SD as the design/clustering component. The two variances are combined, `se_total = sqrt(post_sd² + boot_sd²)`, and the band is `estimate ± 1.96 · se_total` — a single cluster-robust interval, with the posterior median unchanged as the point estimate. One shared set of resamples (`Wb`, a respondent × `boot_B` multiplicity matrix) is reused across every moderator and level for efficiency; the bootstrap runs after `tau_draws` is freed, so it needs only the length-`n` `tau_hat`.

**The verdict is that clustering does not explain the narrowness.** On the real fits the bootstrap SD is roughly an order of magnitude smaller than `post_sd` at every level (e.g. `post_sd` ≈ 0.03–0.08 versus `boot_sd` ≈ 0.003–0.011), so `se_total` exceeds `post_sd` by only about 1–3%. The posterior SD is itself the dominant term and already widens as the level count falls (it scales roughly like 1/√n: a ~40k-row level carries about half the posterior SD of a ~3k-row level). The bands are genuinely that precise — a consequence of how aggressively BCF pools strength across the sample — and the very rare levels that looked suspiciously tight are dropped anyway by the 1% rule. We keep the cluster-robust band because it is the more honest object, but it confirms rather than overturns the posterior intervals.

In [ ]:
#| label: boot-helpers
# build the respondent x B bootstrap multiplicity matrix once per game
make_Wb <- function(rid, B, seed) {
  set.seed(seed)
  rid_int <- as.integer(rid)
  R <- max(rid_int)
  Wb <- vapply(
    seq_len(B),
    function(b) tabulate(sample.int(R, R, replace = TRUE), nbins = R),
    integer(R)
  )
  list(Wb = Wb, rid_int = rid_int)
}

# per-level cluster-bootstrap SD of the bin mean of tau_hat
boot_sd_one <- function(tau_hat, rid_int, Wb, mm, mod) {
  ks <- level_keeps(mm, mod)
  bind_rows(lapply(ks, function(L) {
    keep <- L$keep
    w <- Wb[rid_int[keep], , drop = FALSE] # n_keep x B multiplicities
    bm <- colSums(w * tau_hat[keep]) / colSums(w) # B bootstrap bin means
    tibble(mod = mod, level = L$level, boot_sd = sd(bm))
  }))
}

# join the bootstrap SD and form the cluster-robust interval
finalize_ci <- function(df, boot) {
  df |>
    left_join(boot, by = c("mod", "level", "game")) |>
    mutate(
      boot_sd = coalesce(boot_sd, 0),
      se_total = sqrt(post_sd^2 + boot_sd^2),
      lower = estimate - 1.96 * se_total,
      upper = estimate + 1.96 * se_total
    )
}

## Combined plotters

`plot_comb()` draws the faceted overview with a free y-axis per facet: grey per-level count bars (pooled over games) along the bottom, posterior medians and credible intervals dodged by game everywhere, plus a connecting line and ribbon on the ordered-scale facets only. It is reused verbatim from the combined section of `07` (it reads only the summary columns the BCF builders produce, so the posterior intervals slot straight in).

In [ ]:
#| label: plot-comb
plot_comb <- function(df, ncol, title, ylab = yl_i) {
  d <- df |>
    arrange(mod, order) |>
    mutate(
      key = factor(
        paste0(mod, "@@", level),
        levels = unique(paste0(mod, "@@", level))
      )
    )
  # per-facet grey count bars (pooled over games shown)
  band <- d |>
    summarise(cnt = sum(n), .by = c(mod, key)) |>
    left_join(
      d |>
        summarise(
          floor_y = min(lower) - 0.12 * (max(upper) - min(lower)),
          span = 0.20 * (max(upper) - min(lower)),
          .by = mod
        ),
      by = "mod"
    ) |>
    mutate(bar_top = floor_y + (cnt / max(cnt)) * span, .by = mod)
  d_scale <- dplyr::filter(d, is_scale)
  d_cat <- dplyr::filter(d, !is_scale)
  p <- ggplot(d, aes(x = key, y = estimate, colour = game, shape = game)) +
    geom_segment(
      data = band,
      aes(x = key, xend = key, y = floor_y, yend = bar_top),
      colour = "grey80",
      linewidth = 3,
      inherit.aes = FALSE
    ) +
    ref_zero
  # ordered scales: overlaid ribbon + line + point (continuous-style)
  if (nrow(d_scale) > 0) {
    p <- p +
      geom_ribbon(
        data = d_scale,
        aes(ymin = lower, ymax = upper, fill = game, group = game),
        alpha = 0.15,
        colour = NA
      ) +
      geom_line(data = d_scale, aes(group = game, linetype = game)) +
      geom_point(data = d_scale) +
      scale_fill_manual(values = game_cols, name = NULL)
  }
  # categorical: dodged credible interval + point
  if (nrow(d_cat) > 0) {
    p <- p +
      geom_linerange(
        data = d_cat,
        aes(ymin = lower, ymax = upper, linetype = game),
        position = dodge
      ) +
      geom_point(data = d_cat, position = dodge)
  }
  p +
    facet_wrap(
      ~mod,
      scales = "free",
      ncol = ncol,
      labeller = labeller(mod = mod_strip)
    ) +
    scale_colour_game() +
    scale_shape_game() +
    scale_linetype_game() +
    scale_x_discrete(labels = relabel_key) +
    labs(title = title, x = NULL, y = ylab) +
    guides(fill = "none", linetype = "none") +
    theme_mbeu() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 7))
}

`cont_panel()` draws one genuinely continuous moderator on a continuous x-axis: its per-grid-cell CATE (ribbon + line + point per game) over a grey histogram of the covariate, with the x-axis trimmed to `xlim`. Unlike the `07` version it takes the **precomputed** combined sub-tibble `d` (the BCF draws are already freed by plotting time) and the raw covariate values `raw` for the histogram.

In [ ]:
#| label: cont-panel
cont_panel <- function(d, raw, mod, xlim, ylab = yl_i) {
  # covariate distribution within the visible range
  v <- raw[!is.na(raw) & raw >= xlim[1] & raw <= xlim[2]]
  h <- hist(v, breaks = 30, plot = FALSE)
  floor_y <- min(d$lower) - 0.12 * (max(d$upper) - min(d$lower))
  span <- 0.20 * (max(d$upper) - min(d$lower))
  hd <- tibble(mid = h$mids, w = diff(h$breaks)[1], cnt = h$counts) |>
    mutate(bar_top = floor_y + (cnt / max(cnt)) * span)
  ggplot(d, aes(x = order, y = estimate, colour = game, shape = game)) +
    geom_rect(
      data = hd,
      aes(
        xmin = mid - w / 2,
        xmax = mid + w / 2,
        ymin = floor_y,
        ymax = bar_top
      ),
      fill = "grey80",
      inherit.aes = FALSE
    ) +
    ref_zero +
    geom_ribbon(
      aes(ymin = lower, ymax = upper, fill = game, group = game),
      alpha = 0.15,
      colour = NA
    ) +
    geom_line(aes(group = game, linetype = game)) +
    geom_point() +
    scale_colour_game() +
    scale_shape_game() +
    scale_linetype_game() +
    scale_fill_manual(values = game_cols, name = NULL) +
    scale_x_continuous(breaks = scales::breaks_pretty(6)) +
    coord_cartesian(xlim = xlim) +
    labs(title = mod_labels[[mod]], x = "Covariate value (years)", y = ylab) +
    theme_mbeu()
}

## Build summaries

`tau_draws` is large (`nsim × n`); we extract it one game at a time and free it once the small summary tibbles (ATE, projection, and the combined CATE-by-moderator tables) are built.

In [ ]:
#| label: build-dictator
d_dict <- prep_game(dat, "cj_dict")
m_dict <- game_mods(dat, "cj_dict") |>
  # presentational recodings (display only; sub_X / projection untouched)
  mutate(
    across(
      c(
        q_identity_country,
        q_identity_eu,
        q_identity_europe,
        q_eu_imp_nat_econ,
        q_eu_imp_nat_cul,
        q_eu_imp_nat_pol
      ),
      flip5
    ),
    q_class = factor(q_class, levels = class_order),
    q_rural_urban = factor(
      q_rural_urban,
      levels = c("rural", "small_med_town", "large_city")
    ),
    der_cj_eu_identity = factor(
      der_cj_eu_identity,
      levels = c("eu_citizen", "not_eu_citizen", "not_displayed")
    ),
    der_cj_partisanship = factor(
      der_cj_partisanship,
      levels = c("not_shown", "shown", "not_applicable")
    )
  )
tau_dict <- get_forest_fit(fit_dict$moderate_fit, d_dict$sub_X) # nsim x n

Loading...
ntrees 50
p 55
leaf parameter dimension 1
ndraws 1000
done
Predicting for 84603 new observations

[1] -0.5634977

In [ ]:
#| label: build-trust
d_trust <- prep_game(dat, "cj_trust")
m_trust <- game_mods(dat, "cj_trust") |>
  mutate(
    across(
      c(
        q_identity_country,
        q_identity_eu,
        q_identity_europe,
        q_eu_imp_nat_econ,
        q_eu_imp_nat_cul,
        q_eu_imp_nat_pol
      ),
      flip5
    ),
    q_class = factor(q_class, levels = class_order),
    q_rural_urban = factor(
      q_rural_urban,
      levels = c("rural", "small_med_town", "large_city")
    ),
    der_cj_eu_identity = factor(
      der_cj_eu_identity,
      levels = c("eu_citizen", "not_eu_citizen", "not_displayed")
    ),
    der_cj_partisanship = factor(
      der_cj_partisanship,
      levels = c("not_shown", "shown", "not_applicable")
    )
  )
tau_trust <- get_forest_fit(fit_trust$moderate_fit, d_trust$sub_X) # nsim x n

Loading...
ntrees 50
p 55
leaf parameter dimension 1
ndraws 1000
done
Predicting for 84603 new observations

[1] -0.6451225

The combined tables, with both games stacked and `mod` ordered for faceting.

In [ ]:
#| label: combine
game_lvls <- c("Dictator game", "Trust game")

ate_comb <- bind_rows(
  ate_dict |> mutate(game = "Dictator game"),
  ate_trust |> mutate(game = "Trust game")
) |>
  mutate(game = factor(game, levels = game_lvls))

proj_comb <- bind_rows(
  proj_dict |> mutate(game = "Dictator game"),
  proj_trust |> mutate(game = "Trust game")
) |>
  mutate(game = factor(game, levels = game_lvls))

boot_prof <- bind_rows(boot_prof_dict, boot_prof_trust)
boot_resp <- bind_rows(boot_resp_dict, boot_resp_trust)

comb_prof <- bind_rows(comb_prof_dict, comb_prof_trust) |>
  finalize_ci(boot_prof) |>
  mutate(mod = factor(mod, levels = prof_mods))
comb_resp <- bind_rows(comb_resp_dict, comb_resp_trust) |>
  finalize_ci(boot_resp) |>
  mutate(mod = factor(mod, levels = resp_mods))

stopifnot(
  nrow(ate_comb) == 2,
  all(prof_mods %in% comb_prof$mod),
  all(resp_mods %in% comb_resp$mod),
  # cluster-robust band is never narrower than the posterior-only band
  all(comb_prof$se_total >= comb_prof$post_sd - 1e-9),
  all(comb_resp$se_total >= comb_resp$post_sd - 1e-9)
)

## The recovered ATEs

The posterior ATE — the per-draw average of τ(x) over all observations — for each game, reported as the posterior median and 95% credible interval (the manuscript quotes these numbers in a footnote alongside the GRF SATEs).

In [ ]:
#| label: ate-bcf-numbers
for (i in seq_len(nrow(ate_comb))) {
  cat(sprintf(
    "%s BCF ATE (posterior median): %.4f;  95%% CrI [%.4f, %.4f]\n",
    ate_comb$game[i],
    ate_comb$estimate[i],
    ate_comb$lower[i],
    ate_comb$upper[i]
  ))
}

Dictator game BCF ATE (posterior median): -0.5644;  95% CrI [-0.5983, -0.5273]
Trust game BCF ATE (posterior median): -0.6444;  95% CrI [-0.6822, -0.6089]

## The posterior linear projections

The posterior projection of τ(x) onto the moderator design: a linear summary of which moderators shift the treatment effect, the Bayesian counterpart to GRF’s best linear projection (Woody et al. ([2021](#ref-woody2021model))). This is an appendix/robustness summary, not a headline. Terms are ordered by their mean posterior coefficient across the two games.

In [ ]:
#| label: fig-bcf-combined-proj
#| fig-cap: "BCF posterior linear projection of the CATE onto the moderator set, both games overlaid, with 95% credible intervals (dodged by game). Each coefficient is the posterior of (X'X)^{-1} X' tau_d. Terms are ordered by their mean coefficient across the two games. Appendix-style linear summary of effect modification."
#| dpi: 500
#| fig-width: 6.5
#| fig-height: 9
proj_ord <- proj_comb |>
  summarise(m = mean(estimate), .by = term) |>
  arrange(m) |>
  pull(term)
proj_plot <- proj_comb |> mutate(term = factor(term, levels = proj_ord))

ggplot(proj_plot, aes(x = estimate, y = term, colour = game, shape = game)) +
  geom_vline(xintercept = 0, linetype = "dashed", colour = "grey60") +
  geom_pointrange(aes(xmin = lower, xmax = upper), position = dodge) +
  scale_colour_game() +
  scale_shape_game() +
  labs(
    title = "Posterior linear projection of the CATE, both games",
    x = "Projection coefficient",
    y = NULL
  ) +
  theme_mbeu()

## The posterior CATEs by moderator

The headline figures: posterior median CATE on the y-axis, the moderator on the x-axis, with a cluster-robust 95% interval at each level/grid cell, both games overlaid. The y-axis is free per facet; discrete levels comprising less than 1% of a game’s sample are dropped because their CATE is too imprecise to compare. The three identity scales are reversed so higher means more attachment, the three “EU effect on” items are reversed so higher means a more positive evaluation, and `q_class` runs working class up to higher class with `dont_know` last.

## Conjoint round-level moderators

All five conjoint-round (profile-level) moderators on one canvas, Dictator and Trust overlaid. All five are categorical (including `cj_age`, whose five values are fixed conjoint ages), so each is drawn as dodged points with cluster-robust intervals.

In [ ]:
#| label: fig-bcf-combined-prof
#| fig-cap: "BCF posterior estimates of the profile-level CATE ($\\hat{\\tau}_{ric}$) of the Muslim profile across the levels of every conjoint-round moderator (x-axis), Dictator and Trust games overlaid and distinguished jointly by colour, shape, and linetype. The y-axis is $\\hat{\\tau}_{ric}$ on the token-allocation scale. Each point is a per-level posterior median with a cluster-robust 95% interval, formed as the median plus or minus 1.96 times a combined standard error (a normal approximation) that adds the posterior SD of the within-level mean to a respondent cluster-bootstrap SD. Grey bars along the bottom of each facet show the per-level row count (pooled over games). The y-axis is free per facet. In the Profile partisanship facet, partisanship not shown refers to within the co-national category only, since partisanship is displayed for co-national profiles only."
#| dpi: 500
#| fig-width: 9
#| fig-height: 6
plot_comb(
  comb_prof,
  ncol = 3,
  title = bquote(
    "CATE by conjoint-round moderator, both games" ~ (hat(tau)[ric])
  ),
  ylab = yl_ric
)

## Individual-level moderators

The respondent-level moderators on two figures so the facets stay legible. Each figure stacks (via `patchwork`) a faceted panel of half the discrete moderators on top with one of the two genuinely continuous moderators (`q_age`, `q_edu_age_stop`) on a continuous x-axis below, so the continuous covariate keeps its real histogram and trimmed axis while still sitting beside the discrete moderators it belongs with. Ordered numeric scales (the 5- and 7-point items) carry an overlaid cluster-robust 95% ribbon plus connecting line per game; categorical moderators keep dodged points and intervals. The three identity scales are reversed so higher means more attachment, the three “EU effect on” items are reversed so higher means a more positive evaluation, and `q_class` runs working class up to higher class with `dont_know` last. The shared game legend is drawn once, from the top panel.

In [ ]:
#| label: resp-split
resp_cat <- setdiff(resp_mods, c("q_age", "q_edu_age_stop"))
# categorical moderators first, then the ordered scales
resp_cat <- c(intersect(cat_mods, resp_cat), setdiff(resp_cat, cat_mods))
half <- ceiling(length(resp_cat) / 2)
resp_half1 <- resp_cat[1:half]
resp_half2 <- resp_cat[(half + 1):length(resp_cat)]

In [ ]:
#| label: fig-bcf-combined-resp-1
#| fig-cap: "BCF posterior estimates of the respondent-level CATE ($\\hat{\\tau}_i$) of the Muslim profile across the levels of the first half of the discrete individual-level moderators (top panel) and across age (q_age, bottom panel), Dictator and Trust games overlaid and distinguished jointly by colour, shape, and linetype. The y-axis is $\\hat{\\tau}_i$ on the token-allocation scale. All item scales are oriented so that higher x-axis values mean more agreement, a more positive evaluation, or more attachment, depending on the item (see @tbl-wordings). Every point and ribbon value is a posterior median with a cluster-robust 95% interval, formed as the median plus or minus 1.96 times a combined standard error (a normal approximation) that adds the posterior SD to a respondent cluster-bootstrap SD. Top: ordered numeric scales are drawn as a connecting line and interval ribbon per game, categorical moderators as dodged points and intervals. Grey bars show the per-level row count (pooled over games). Bottom: per-grid-cell estimate, interval ribbon, and connecting line per game over a grey histogram of the covariate, on 100 equal intervals across the full 18-75 years. The y-axis is free per facet. Discrete levels comprising less than 1% of a game's sample are dropped because their CATE is too imprecise to compare."
#| dpi: 500
#| fig-width: 11
#| fig-height: 14
p_resp1 <- plot_comb(
  comb_resp |>
    filter(mod %in% resp_half1) |>
    mutate(mod = factor(mod, levels = resp_half1)),
  ncol = 4,
  title = bquote(
    "CATE by individual-level moderator, both games (1 of 2)" ~ (hat(tau)[ic])
  )
)
p_age <- cont_panel(
  comb_resp |> filter(mod == "q_age"),
  m_dict[["q_age"]],
  "q_age",
  c(18, 75)
) +
  guides(colour = "none", shape = "none", fill = "none", linetype = "none")
(p_resp1 / p_age) + plot_layout(heights = c(3, 1))

In [ ]:
#| label: fig-bcf-combined-resp-2
#| fig-cap: "BCF posterior estimates of the respondent-level CATE ($\\hat{\\tau}_i$) of the Muslim profile across the levels of the second half of the discrete individual-level moderators (top panel) and across age at which education stopped (q_edu_age_stop, bottom panel), Dictator and Trust games overlaid and distinguished jointly by colour, shape, and linetype. The y-axis is $\\hat{\\tau}_i$ on the token-allocation scale. All item scales are oriented so that higher x-axis values mean more agreement, a more positive evaluation, or more attachment, depending on the item (see @tbl-wordings). Every ribbon value is a posterior median with a cluster-robust 95% interval, formed as the median plus or minus 1.96 times a combined standard error (a normal approximation) that adds the posterior SD to a respondent cluster-bootstrap SD. Top: ordered numeric scales are drawn as a connecting line and interval ribbon per game. Grey bars show the per-level row count (pooled over games). Bottom: per-year posterior median, interval ribbon, and connecting line per game at each integer year of education-stopping age from 15 to 30, over a grey histogram of the covariate. The y-axis is free per facet. Discrete levels comprising less than 1% of a game's sample are dropped because their CATE is too imprecise to compare."
#| dpi: 500
#| fig-width: 11
#| fig-height: 14
p_resp2 <- plot_comb(
  comb_resp |>
    filter(mod %in% resp_half2) |>
    mutate(mod = factor(mod, levels = resp_half2)),
  ncol = 4,
  title = bquote(
    "CATE by individual-level moderator, both games (2 of 2)" ~ (hat(tau)[ic])
  )
)
p_edu <- cont_panel(
  comb_resp |> filter(mod == "q_edu_age_stop"),
  m_dict[["q_edu_age_stop"]],
  "q_edu_age_stop",
  c(15, 30)
) +
  guides(colour = "none", shape = "none", fill = "none", linetype = "none")
(p_resp2 / p_edu) + plot_layout(heights = c(3, 1))

## Session Info

In [ ]:
#| label: session-info
#| eval: true
#| echo: true
session_info()

─ Session info ───────────────────────────────────────────────────────────────
 setting  value
 version  R version 4.5.3 (2026-03-11 ucrt)
 os       Windows 11 x64 (build 26200)
 system   x86_64, mingw32
 ui       RTerm
 language (EN)
 collate  English_United States.utf8
 ctype    English_United States.utf8
 tz       Europe/Berlin
 date     2026-06-10
 pandoc   3.8.3 @ c:\\Program Files\\Positron\\resources\\app\\quarto\\bin\\tools/ (via rmarkdown)
 quarto   1.9.36 @ C:\\PROGRA~1\\Quarto\\bin\\quarto.exe

─ Packages ───────────────────────────────────────────────────────────────────
 package       * version    date (UTC) lib source
 abind           1.4-8      2024-09-12 [1] CRAN (R 4.5.0)
 backports       1.5.0      2024-05-23 [1] CRAN (R 4.5.0)
 broom           1.0.11     2025-12-04 [1] CRAN (R 4.5.2)
 car             3.1-3      2024-09-27 [1] CRAN (R 4.5.1)
 carData         3.0-5      2022-01-06 [1] CRAN (R 4.5.1)
 cli             3.6.5      2025-04-23 [1] CRAN (R 4.5.1)
 codetools  

## Execution Time

In [ ]:
#| label: exec-time
#| eval: true
#| echo: true
#| include: true
end_time <- Sys.time()
exec_time <- end_time - start_time
cat(paste(
  "R code execution time:",
  round(as.numeric(exec_time, units = "secs"), 2),
  "seconds."
))

R code execution time: 524 seconds.

```` markdown
---
title: "BCF Post-Processing"
subtitle: |
  BCF Post-Processing — ATE, Posterior Projection, and CATE by Moderator (Dictator + Trust)
date: last-modified
date-format: MMMM D, YYYY
format:
  html:
    toc: true
    toc-depth: 3
    code-fold: true
    code-tools: true
execute:
  echo: true
  warning: true
  eval: true
  message: true
  cache: true
---

## Purpose {#sec-purpose}

This notebook post-processes the **robustness** estimator — the saved hierarchical BCF fits
(`data/03_final/bcf_{dictator,trust}.rds`) — into the same three quantities, in the same order,
as the combined view of the primary GRF notebook `07_postprocess_grf.qmd`:

1. **Average treatment effect (ATE)** — the posterior of the ATE for each game.
2. **Posterior linear projection** — the Bayesian counterpart to GRF's best linear projection
   (`woody2021model`): each posterior draw of τ(x) is projected onto the moderator design, giving
   a posterior distribution of projection coefficients. Appendix-style linear summary.
3. **CATE by moderator** — the headline figures: estimated CATE on the y-axis, a moderator on the
   x-axis.

It is the BCF counterpart to `07_postprocess_grf.qmd` and is structurally parallel to it; only the
inference machinery differs. GRF reports point estimates with honest cluster-robust confidence
intervals; BCF reports **posterior medians** (read off the τ(x) draws from `get_forest_fit()`) with
95% credible intervals for the ATE and projection, and a cluster-robust band for the CATE-by-moderator
figures that adds a respondent cluster bootstrap to the posterior SD (see @sec-cluster-bootstrap).

Following the **combined-figure** grammar of `07` (its `# Combined figures` section) and the house
style of `09_additional_figs_tables.qmd`, every figure overlays **both games on one canvas**. The two
games are the only thing encoded by colour, shape, and linetype, and they are always encoded by all
three together, so each game is distinguishable in colour, in greyscale print, and under
colourblindness. Every other distinction (which moderator, which level) is carried by facets and
axis position.

Two moderator families, matching the research plan:

- **Conjoint round-level** moderators (`prof_mods`) — other attributes in the same profile.
- **Individual-level** moderators (`resp_mods`) — respondent pre-treatment items and indices.

Per-level uncertainty is a **cluster-robust 95% interval**: the posterior median of the bin/level
CATE (cf. `fig-cate-by-cov-bcf` in `03_multibart_nested_ri_test.qmd`) with a band that combines the
posterior SD (model/estimation uncertainty) and a **respondent cluster bootstrap** SD (within-person
dependence across the three rounds; see @sec-cluster-bootstrap). We added the bootstrap to test the
worry that the bands looked too narrow at low-count levels: the answer is that within-respondent
clustering does **not** explain it — the posterior SD dominates the bootstrap SD by roughly an order
of magnitude at every level, and it already widens appropriately as the level count shrinks. The
bands are, in other words, genuinely that precise; the bootstrap is folded in for completeness but
moves them by only a few percent.

## Setup {#sec-setup}

The setup chunk loads the project packages plus `multibart`'s own dependencies before the local
package is attached (needed for `get_forest_fit()`).

quarto-executable-code-5450563D

```r
#| label: setup
#| include: false
# execution time
start_time <- Sys.time()
# console width
options(width = 80)
# packages
p_required <- c(
  "tidyverse", # dplyr, ggplot2, tidyr, purrr
  "here", # relative paths
  "ggpubr", # theme_pubr
  "patchwork", # combined-figure composition
  "sessioninfo", # session docs

  # multibart dependencies -----------------------------------------------------
  "Rcpp", # multibart Depends + LinkingTo
  "RcppArmadillo", # multibart Depends + LinkingTo
  "Rcereal", # multibart Depends + LinkingTo
  "fastDummies" # multibart helper dependency
)
packages <- rownames(installed.packages())
p_to_install <- p_required[!(p_required %in% packages)]
if (length(p_to_install) > 0) {
  install.packages(p_to_install)
}
sapply(p_required, require, character.only = TRUE)
rm(p_required, p_to_install, packages)
```

quarto-executable-code-5450563D

```r
#| label: setup-multibart
#| include: false
# install local source on first use
if (!requireNamespace("multibart", quietly = TRUE)) {
  install.packages(
    here::here("code", "multibart"),
    repos = NULL,
    type = "source"
  )
}
# attach installed package
library(multibart)
```

## Load data and fits {#sec-load}

quarto-executable-code-5450563D

```r
#| label: load-data
#| cache: false
dat <- readRDS(here::here("data", "02_processed", "eu25_long.rds"))
fit_dict <- readRDS(here::here("data", "03_final", "bcf_dictator.rds"))
fit_trust <- readRDS(here::here("data", "03_final", "bcf_trust.rds"))
```

## Variable definitions {#sec-vars}

The frozen moderator list and `prep_game()` are reused verbatim from `05_bcf_fit.qmd` so the
reconstructed moderator design matches fit-time row order and columns exactly.

quarto-executable-code-5450563D

```r
#| label: moderators
prof_mods <- c(
  "cj_nat",
  "der_cj_eu_identity",
  "der_cj_partisanship",
  "cj_age",
  "cj_sex",
  "cj_class"
)
resp_mods <- c(
  "q_gender",
  "q_age",
  "q_identity_country",
  "q_identity_eu",
  "q_identity_europe",
  "q_religion",
  "q_class",
  "q_eu_efficacy_understand",
  "q_pop_reps",
  "q_pop_goodevil",
  "q_pop_compromise",
  "q_dem_compromise",
  "q_dem_listen",
  "q_tech_experts",
  "q_tech_leaders",
  "q_party_harm",
  "q_people_incompetent",
  "q_eu_longterm",
  "q_eu_responsive",
  "q_eu_imp_nat_econ",
  "q_eu_imp_nat_cul",
  "q_eu_imp_nat_pol",
  "q_eu_abolish",
  "q_eu_satisfaction",
  "q_rural_urban",
  "q_edu_age_stop"
)
```

quarto-executable-code-5450563D

```r
#| label: prep-game
prep_game <- function(dat, game) {
  sub <- dat |>
    filter(cj_game_type == game) # full per-game sample (no subsampling)

  y <- sub$cj_pl2
  z <- as.integer(sub$cj_rel == "muslim") # Muslim vs all-else pooled
  country_id <- factor(sub$meta_country)
  respondent_id <- factor(sub$meta_pid)

  mod_df <- sub |>
    select(all_of(c(prof_mods, resp_mods))) |>
    mutate(across(where(is.character), factor))
  sub_X <- model.matrix(~ . - 1, data = mod_df) # factors -> dummies

  stopifnot(
    !anyNA(sub_X),
    all(z %in% c(0L, 1L)),
    !anyNA(y),
    nrow(sub_X) == length(y)
  )

  list(
    y = y,
    z = z,
    sub_X = sub_X,
    country_id = country_id,
    respondent_id = respondent_id
  )
}
```

`game_mods()` returns the original (pre-dummy) moderator values for one game, in the same row
order as `prep_game()`, so each observation's τ̂ aligns with its moderator value.

quarto-executable-code-5450563D

```r
#| label: game-mods
game_mods <- function(dat, game) {
  dat |>
    filter(cj_game_type == game) |>
    select(all_of(c(prof_mods, resp_mods))) |>
    mutate(across(where(is.character), factor))
}
```

## House style {#sec-house}

The Okabe-Ito game palette with redundant shape and linetype, the three reserved scales, the
shared theme, and the reference layers — copied from `09_additional_figs_tables.qmd` / the combined
section of `07_postprocess_grf.qmd` so the BCF figures share the manuscript grammar exactly.

quarto-executable-code-5450563D

```r
#| label: game-style
game_labs <- c(cj_dict = "Dictator game", cj_trust = "Trust game")
game_cols <- c("Dictator game" = "#0072B2", "Trust game" = "#D55E00")
game_shapes <- c("Dictator game" = 16, "Trust game" = 17) # circle / triangle
game_ltys <- c("Dictator game" = "solid", "Trust game" = "longdash")

scale_colour_game <- function() {
  scale_colour_manual(values = game_cols, name = NULL)
}
scale_shape_game <- function() {
  scale_shape_manual(values = game_shapes, name = NULL)
}
scale_linetype_game <- function() {
  scale_linetype_manual(values = game_ltys, name = NULL)
}

# shared moderator labels + categorical-first ordering for the facets
source(here::here("code", "helper_scripts", "moderator_labels.R"))

theme_mbeu <- function(base_size = 11) {
  ggpubr::theme_pubr(base_size = base_size) +
    theme(legend.position = "top")
}
ref_zero <- geom_hline(yintercept = 0, linetype = "dashed", colour = "grey60")
dodge <- position_dodge(width = 0.5)
yl_cate <- expression(widehat(CATE))
yl_ric <- expression(hat(tau)[ric]) # profile-level CATE
yl_i <- expression(hat(tau)[ic]) # respondent-level CATE
```

## Posterior summary builders {#sec-machinery}

Every quantity below is read off `tau_draws`, the `nsim × n` posterior matrix returned by
`get_forest_fit()` (rows index posterior draws, columns index observations). All three builders
operate on a single game's `tau_draws`, so they run inside the per-game block and the large matrix
is freed before the next game.

A moderator is **discrete** (factor, or numeric with ≤ 10 distinct values: Likert items) or
**binned** (higher-cardinality numeric → 10 quantile bins).

quarto-executable-code-5450563D

```r
#| label: classify
classify_mod <- function(col) {
  if (is.factor(col) || is.character(col)) {
    return("discrete")
  }
  if (dplyr::n_distinct(col) <= 10) {
    return("discrete")
  }
  "binned"
}
```

Shared display specs, matching `07_postprocess_grf.qmd`: the substantive `q_class` order
(`dont_know` last), the 5-point-scale reversal (`6 - x`) applied to the identity scales (so higher
means more attachment) and to the "EU effect on" items (so higher means a more positive evaluation),
the two fixed display grids for the genuinely continuous moderators, and the number of cluster
bootstrap resamples.

quarto-executable-code-5450563D

```r
#| label: display-specs
class_order <- c(
  "working_class",
  "lower_middle_class",
  "middle_class",
  "upper_middle_class",
  "higher_class",
  "dont_know",
  "other"
)
flip5 <- \(x) 6 - x # 5-point scales: reverse so higher = more of the construct
cont_grids <- list(
  q_age = seq(18, 75, length.out = 101), # 100 intervals over the full range
  q_edu_age_stop = seq(14.5, 30.5, by = 1) # integer years 15..30
)
boot_B <- 500L # respondent cluster bootstrap resamples
```

`ate_summary()` returns the posterior of the ATE: the per-draw mean of τ(x) over all observations,
summarised by its median and a 95% credible interval.

quarto-executable-code-5450563D

```r
#| label: ate-summary
ate_summary <- function(tau_draws) {
  bd <- rowMeans(tau_draws) # posterior of the ATE
  tibble(
    estimate = median(bd),
    lower = unname(quantile(bd, 0.025)),
    upper = unname(quantile(bd, 0.975))
  )
}
```

`proj_summary()` is the Bayesian posterior projection (`woody2021model`), the BCF counterpart to
GRF's `best_linear_projection`. For each posterior draw `d`, the coefficient vector is
$\beta_d = (X'X)^{-1} X' \tau_d$; we stack these into a posterior and summarise each term by its
median and 95% credible interval.
The design `sub_X` is `model.matrix(~ . - 1, ...)`, which is full rank (only the first factor is
fully expanded; the rest are contrast-coded), so `crossprod(sub_X)` inverts directly — no added
intercept, no ridge. It is the same column basis as GRF's `X_dict`, so the terms are comparable.
The only large multiply (`tau_draws %*% sub_X`) returns a small `nsim × p` matrix, so the step is
memory-safe.

quarto-executable-code-5450563D

```r
#| label: proj-summary
proj_summary <- function(tau_draws, sub_X) {
  xtx <- crossprod(sub_X) # p x p
  m <- tau_draws %*% sub_X # nsim x p  (= tau_d' X per draw)
  beta <- solve(xtx, t(m)) # p x nsim  = (X'X)^{-1} X' tau_d
  tibble(
    term = colnames(sub_X),
    estimate = apply(beta, 1, median),
    lower = apply(beta, 1, quantile, 0.025),
    upper = apply(beta, 1, quantile, 0.975)
  )
}
```

`level_keeps()` enumerates the kept levels/bins of one moderator and returns, for each, the logical
keep-vector over the game's rows together with its display `level`, `order`, and `is_scale` tag.
It mirrors the combined builder in `07` exactly — discrete levels honor the factor order and are
dropped below **1% of the game sample** (their CATE is too imprecise to compare); `cj_age` is
treated as categorical (fixed conjoint ages); and the two genuinely continuous moderators are
binned on the **fixed display grid** (`cont_grids`) rather than into deciles, so the line spans the
full range. Sharing this enumeration between the posterior summary and the cluster bootstrap
guarantees the two use identical bins.

quarto-executable-code-5450563D

```r
#| label: level-keeps
level_keeps <- function(mm, mod) {
  v <- mm[[mod]]
  is_num <- is.numeric(v)
  # genuinely continuous moderators -> fixed grid (or decile fallback)
  if (is_num && length(unique(v)) > 12) {
    if (mod %in% names(cont_grids)) {
      br <- cont_grids[[mod]]
      cell <- cut(v, breaks = br, include.lowest = TRUE)
      mids <- (head(br, -1) + tail(br, -1)) / 2
      out <- lapply(
        which(tabulate(cell, nbins = length(mids)) > 0),
        function(k) {
          keep <- !is.na(cell) & as.integer(cell) == k
          list(
            level = as.character(round(mids[k])),
            order = mids[k],
            is_scale = TRUE,
            keep = keep
          )
        }
      )
      return(out)
    }
    bin <- ntile(v, 10)
    return(lapply(sort(unique(bin)), function(k) {
      keep <- !is.na(bin) & bin == k
      list(
        level = as.character(round(mean(v[keep]))),
        order = mean(v[keep]),
        is_scale = TRUE,
        keep = keep
      )
    }))
  }
  # discrete moderators: honor factor order; drop levels below 1% of the game sample
  levs <- if (is_num) {
    sort(unique(v))
  } else if (is.factor(v)) {
    levels(droplevels(v))
  } else {
    sort(unique(as.character(v)))
  }
  out <- lapply(seq_along(levs), function(i) {
    lev <- levs[i]
    keep <- if (is_num) {
      !is.na(v) & v == lev
    } else {
      !is.na(v) & as.character(v) == lev
    }
    if (sum(keep) / sum(!is.na(v)) < 0.01) {
      return(NULL)
    }
    list(
      level = as.character(lev),
      order = if (is_num) as.numeric(lev) else i,
      is_scale = is_num && mod != "cj_age",
      keep = keep
    )
  })
  out[!vapply(out, is.null, logical(1))]
}
```

`cate_one_bcf()` reads each bin's estimate off the **posterior of the bin mean** (`rowMeans` of
τ(x) over the bin's observations): the point is the posterior median and `post_sd` is the posterior
standard deviation, the **model/estimation** uncertainty. The final interval (built downstream)
adds the respondent-cluster bootstrap variance to this.

quarto-executable-code-5450563D

```r
#| label: cate-one-bcf
# posterior of the bin CATE over a logical keep-vector (length n)
.bin_post <- function(tau_draws, keep) {
  bd <- rowMeans(tau_draws[, keep, drop = FALSE]) # posterior of bin mean
  c(estimate = median(bd), post_sd = sd(bd))
}

cate_one_bcf <- function(tau_draws, mm, mod, game_label) {
  ks <- level_keeps(mm, mod)
  bind_rows(lapply(ks, function(L) {
    p <- .bin_post(tau_draws, L$keep)
    tibble(
      mod = mod,
      level = L$level,
      order = L$order,
      is_scale = L$is_scale,
      game = game_label,
      n = sum(L$keep),
      estimate = unname(p["estimate"]),
      post_sd = unname(p["post_sd"])
    )
  }))
}
```

## Respondent cluster bootstrap {#sec-cluster-bootstrap}

The posterior `post_sd` above captures **model/estimation** uncertainty in the regularized τ(x)
surface alone — not the within-respondent design dependence (each respondent contributes three
conjoint rounds) and not the sampling variability of which respondents land in a bin. The worry was
that omitting this clustering made the bands look too narrow, especially at sparsely populated
levels. We test that directly with a **respondent cluster bootstrap**: resample respondents (the
clusters) with replacement `boot_B` times, recompute each bin's mean of the per-observation point
estimate `tau_hat`, and take the bootstrap SD as the design/clustering component. The two variances
are combined, `se_total = sqrt(post_sd² + boot_sd²)`, and the band is `estimate ± 1.96 · se_total` —
a single cluster-robust interval, with the posterior median unchanged as the point estimate. One
shared set of resamples (`Wb`, a respondent × `boot_B` multiplicity matrix) is reused across every
moderator and level for efficiency; the bootstrap runs after `tau_draws` is freed, so it needs only
the length-`n` `tau_hat`.

**The verdict is that clustering does not explain the narrowness.** On the real fits the bootstrap SD
is roughly an order of magnitude smaller than `post_sd` at every level (e.g. `post_sd` ≈ 0.03–0.08
versus `boot_sd` ≈ 0.003–0.011), so `se_total` exceeds `post_sd` by only about 1–3%. The posterior
SD is itself the dominant term and already widens as the level count falls (it scales roughly like
1/√n: a ~40k-row level carries about half the posterior SD of a ~3k-row level). The bands are
genuinely that precise — a consequence of how aggressively BCF pools strength across the sample — and
the very rare levels that looked suspiciously tight are dropped anyway by the 1% rule. We keep the
cluster-robust band because it is the more honest object, but it confirms rather than overturns the
posterior intervals.

quarto-executable-code-5450563D

```r
#| label: boot-helpers
# build the respondent x B bootstrap multiplicity matrix once per game
make_Wb <- function(rid, B, seed) {
  set.seed(seed)
  rid_int <- as.integer(rid)
  R <- max(rid_int)
  Wb <- vapply(
    seq_len(B),
    function(b) tabulate(sample.int(R, R, replace = TRUE), nbins = R),
    integer(R)
  )
  list(Wb = Wb, rid_int = rid_int)
}

# per-level cluster-bootstrap SD of the bin mean of tau_hat
boot_sd_one <- function(tau_hat, rid_int, Wb, mm, mod) {
  ks <- level_keeps(mm, mod)
  bind_rows(lapply(ks, function(L) {
    keep <- L$keep
    w <- Wb[rid_int[keep], , drop = FALSE] # n_keep x B multiplicities
    bm <- colSums(w * tau_hat[keep]) / colSums(w) # B bootstrap bin means
    tibble(mod = mod, level = L$level, boot_sd = sd(bm))
  }))
}

# join the bootstrap SD and form the cluster-robust interval
finalize_ci <- function(df, boot) {
  df |>
    left_join(boot, by = c("mod", "level", "game")) |>
    mutate(
      boot_sd = coalesce(boot_sd, 0),
      se_total = sqrt(post_sd^2 + boot_sd^2),
      lower = estimate - 1.96 * se_total,
      upper = estimate + 1.96 * se_total
    )
}
```

## Combined plotters {#sec-plotters}

`plot_comb()` draws the faceted overview with a free y-axis per facet: grey per-level count bars
(pooled over games) along the bottom, posterior medians and credible intervals dodged by game
everywhere, plus a connecting line and ribbon on the ordered-scale facets only.
It is reused verbatim from the combined section of `07` (it reads only the summary columns the BCF
builders produce, so the posterior intervals slot straight in).

quarto-executable-code-5450563D

```r
#| label: plot-comb
plot_comb <- function(df, ncol, title, ylab = yl_i) {
  d <- df |>
    arrange(mod, order) |>
    mutate(
      key = factor(
        paste0(mod, "@@", level),
        levels = unique(paste0(mod, "@@", level))
      )
    )
  # per-facet grey count bars (pooled over games shown)
  band <- d |>
    summarise(cnt = sum(n), .by = c(mod, key)) |>
    left_join(
      d |>
        summarise(
          floor_y = min(lower) - 0.12 * (max(upper) - min(lower)),
          span = 0.20 * (max(upper) - min(lower)),
          .by = mod
        ),
      by = "mod"
    ) |>
    mutate(bar_top = floor_y + (cnt / max(cnt)) * span, .by = mod)
  d_scale <- dplyr::filter(d, is_scale)
  d_cat <- dplyr::filter(d, !is_scale)
  p <- ggplot(d, aes(x = key, y = estimate, colour = game, shape = game)) +
    geom_segment(
      data = band,
      aes(x = key, xend = key, y = floor_y, yend = bar_top),
      colour = "grey80",
      linewidth = 3,
      inherit.aes = FALSE
    ) +
    ref_zero
  # ordered scales: overlaid ribbon + line + point (continuous-style)
  if (nrow(d_scale) > 0) {
    p <- p +
      geom_ribbon(
        data = d_scale,
        aes(ymin = lower, ymax = upper, fill = game, group = game),
        alpha = 0.15,
        colour = NA
      ) +
      geom_line(data = d_scale, aes(group = game, linetype = game)) +
      geom_point(data = d_scale) +
      scale_fill_manual(values = game_cols, name = NULL)
  }
  # categorical: dodged credible interval + point
  if (nrow(d_cat) > 0) {
    p <- p +
      geom_linerange(
        data = d_cat,
        aes(ymin = lower, ymax = upper, linetype = game),
        position = dodge
      ) +
      geom_point(data = d_cat, position = dodge)
  }
  p +
    facet_wrap(
      ~mod,
      scales = "free",
      ncol = ncol,
      labeller = labeller(mod = mod_strip)
    ) +
    scale_colour_game() +
    scale_shape_game() +
    scale_linetype_game() +
    scale_x_discrete(labels = relabel_key) +
    labs(title = title, x = NULL, y = ylab) +
    guides(fill = "none", linetype = "none") +
    theme_mbeu() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 7))
}
```

`cont_panel()` draws one genuinely continuous moderator on a continuous x-axis: its per-grid-cell
CATE (ribbon + line + point per game) over a grey histogram of the covariate, with the x-axis
trimmed to `xlim`. Unlike the `07` version it takes the **precomputed** combined sub-tibble `d`
(the BCF draws are already freed by plotting time) and the raw covariate values `raw` for the
histogram.

quarto-executable-code-5450563D

```r
#| label: cont-panel
cont_panel <- function(d, raw, mod, xlim, ylab = yl_i) {
  # covariate distribution within the visible range
  v <- raw[!is.na(raw) & raw >= xlim[1] & raw <= xlim[2]]
  h <- hist(v, breaks = 30, plot = FALSE)
  floor_y <- min(d$lower) - 0.12 * (max(d$upper) - min(d$lower))
  span <- 0.20 * (max(d$upper) - min(d$lower))
  hd <- tibble(mid = h$mids, w = diff(h$breaks)[1], cnt = h$counts) |>
    mutate(bar_top = floor_y + (cnt / max(cnt)) * span)
  ggplot(d, aes(x = order, y = estimate, colour = game, shape = game)) +
    geom_rect(
      data = hd,
      aes(
        xmin = mid - w / 2,
        xmax = mid + w / 2,
        ymin = floor_y,
        ymax = bar_top
      ),
      fill = "grey80",
      inherit.aes = FALSE
    ) +
    ref_zero +
    geom_ribbon(
      aes(ymin = lower, ymax = upper, fill = game, group = game),
      alpha = 0.15,
      colour = NA
    ) +
    geom_line(aes(group = game, linetype = game)) +
    geom_point() +
    scale_colour_game() +
    scale_shape_game() +
    scale_linetype_game() +
    scale_fill_manual(values = game_cols, name = NULL) +
    scale_x_continuous(breaks = scales::breaks_pretty(6)) +
    coord_cartesian(xlim = xlim) +
    labs(title = mod_labels[[mod]], x = "Covariate value (years)", y = ylab) +
    theme_mbeu()
}
```

## Build summaries {#sec-build}

`tau_draws` is large (`nsim × n`); we extract it one game at a time and free it once the small
summary tibbles (ATE, projection, and the combined CATE-by-moderator tables) are built.

quarto-executable-code-5450563D

```r
#| label: build-dictator
d_dict <- prep_game(dat, "cj_dict")
m_dict <- game_mods(dat, "cj_dict") |>
  # presentational recodings (display only; sub_X / projection untouched)
  mutate(
    across(
      c(
        q_identity_country,
        q_identity_eu,
        q_identity_europe,
        q_eu_imp_nat_econ,
        q_eu_imp_nat_cul,
        q_eu_imp_nat_pol
      ),
      flip5
    ),
    q_class = factor(q_class, levels = class_order),
    q_rural_urban = factor(
      q_rural_urban,
      levels = c("rural", "small_med_town", "large_city")
    ),
    der_cj_eu_identity = factor(
      der_cj_eu_identity,
      levels = c("eu_citizen", "not_eu_citizen", "not_displayed")
    ),
    der_cj_partisanship = factor(
      der_cj_partisanship,
      levels = c("not_shown", "shown", "not_applicable")
    )
  )
tau_dict <- get_forest_fit(fit_dict$moderate_fit, d_dict$sub_X) # nsim x n

stopifnot(
  nrow(m_dict) == length(d_dict$y),
  ncol(tau_dict) == length(d_dict$y)
)

# per-observation point estimate for the cluster bootstrap
tau_hat_dict <- colMeans(tau_dict)
# ATE sanity check (token scale)
mean(tau_hat_dict)

ate_dict <- ate_summary(tau_dict)
proj_dict <- proj_summary(tau_dict, d_dict$sub_X)
comb_prof_dict <- bind_rows(
  lapply(prof_mods, \(m) cate_one_bcf(tau_dict, m_dict, m, "Dictator game"))
)
comb_resp_dict <- bind_rows(
  lapply(resp_mods, \(m) cate_one_bcf(tau_dict, m_dict, m, "Dictator game"))
)

rm(tau_dict, fit_dict)
invisible(gc())

# respondent cluster bootstrap (runs after tau_dict is freed)
bw_dict <- make_Wb(d_dict$respondent_id, B = boot_B, seed = 1L)
boot_prof_dict <- bind_rows(
  lapply(prof_mods, \(m) {
    boot_sd_one(tau_hat_dict, bw_dict$rid_int, bw_dict$Wb, m_dict, m)
  })
) |>
  mutate(game = "Dictator game")
boot_resp_dict <- bind_rows(
  lapply(resp_mods, \(m) {
    boot_sd_one(tau_hat_dict, bw_dict$rid_int, bw_dict$Wb, m_dict, m)
  })
) |>
  mutate(game = "Dictator game")
rm(bw_dict)
invisible(gc())
```

quarto-executable-code-5450563D

```r
#| label: build-trust
d_trust <- prep_game(dat, "cj_trust")
m_trust <- game_mods(dat, "cj_trust") |>
  mutate(
    across(
      c(
        q_identity_country,
        q_identity_eu,
        q_identity_europe,
        q_eu_imp_nat_econ,
        q_eu_imp_nat_cul,
        q_eu_imp_nat_pol
      ),
      flip5
    ),
    q_class = factor(q_class, levels = class_order),
    q_rural_urban = factor(
      q_rural_urban,
      levels = c("rural", "small_med_town", "large_city")
    ),
    der_cj_eu_identity = factor(
      der_cj_eu_identity,
      levels = c("eu_citizen", "not_eu_citizen", "not_displayed")
    ),
    der_cj_partisanship = factor(
      der_cj_partisanship,
      levels = c("not_shown", "shown", "not_applicable")
    )
  )
tau_trust <- get_forest_fit(fit_trust$moderate_fit, d_trust$sub_X) # nsim x n

stopifnot(
  nrow(m_trust) == length(d_trust$y),
  ncol(tau_trust) == length(d_trust$y)
)

tau_hat_trust <- colMeans(tau_trust)
mean(tau_hat_trust)

ate_trust <- ate_summary(tau_trust)
proj_trust <- proj_summary(tau_trust, d_trust$sub_X)
comb_prof_trust <- bind_rows(
  lapply(prof_mods, \(m) cate_one_bcf(tau_trust, m_trust, m, "Trust game"))
)
comb_resp_trust <- bind_rows(
  lapply(resp_mods, \(m) cate_one_bcf(tau_trust, m_trust, m, "Trust game"))
)

rm(tau_trust, fit_trust)
invisible(gc())

bw_trust <- make_Wb(d_trust$respondent_id, B = boot_B, seed = 2L)
boot_prof_trust <- bind_rows(
  lapply(prof_mods, \(m) {
    boot_sd_one(tau_hat_trust, bw_trust$rid_int, bw_trust$Wb, m_trust, m)
  })
) |>
  mutate(game = "Trust game")
boot_resp_trust <- bind_rows(
  lapply(resp_mods, \(m) {
    boot_sd_one(tau_hat_trust, bw_trust$rid_int, bw_trust$Wb, m_trust, m)
  })
) |>
  mutate(game = "Trust game")
rm(bw_trust)
invisible(gc())
```

The combined tables, with both games stacked and `mod` ordered for faceting.

quarto-executable-code-5450563D

```r
#| label: combine
game_lvls <- c("Dictator game", "Trust game")

ate_comb <- bind_rows(
  ate_dict |> mutate(game = "Dictator game"),
  ate_trust |> mutate(game = "Trust game")
) |>
  mutate(game = factor(game, levels = game_lvls))

proj_comb <- bind_rows(
  proj_dict |> mutate(game = "Dictator game"),
  proj_trust |> mutate(game = "Trust game")
) |>
  mutate(game = factor(game, levels = game_lvls))

boot_prof <- bind_rows(boot_prof_dict, boot_prof_trust)
boot_resp <- bind_rows(boot_resp_dict, boot_resp_trust)

comb_prof <- bind_rows(comb_prof_dict, comb_prof_trust) |>
  finalize_ci(boot_prof) |>
  mutate(mod = factor(mod, levels = prof_mods))
comb_resp <- bind_rows(comb_resp_dict, comb_resp_trust) |>
  finalize_ci(boot_resp) |>
  mutate(mod = factor(mod, levels = resp_mods))

stopifnot(
  nrow(ate_comb) == 2,
  all(prof_mods %in% comb_prof$mod),
  all(resp_mods %in% comb_resp$mod),
  # cluster-robust band is never narrower than the posterior-only band
  all(comb_prof$se_total >= comb_prof$post_sd - 1e-9),
  all(comb_resp$se_total >= comb_resp$post_sd - 1e-9)
)
```

## The recovered ATEs {#sec-ate}

The posterior ATE — the per-draw average of τ(x) over all observations — for each game, reported as
the posterior median and 95% credible interval (the manuscript quotes these numbers in a footnote
alongside the GRF SATEs).

quarto-executable-code-5450563D

```r
#| label: ate-bcf-numbers
for (i in seq_len(nrow(ate_comb))) {
  cat(sprintf(
    "%s BCF ATE (posterior median): %.4f;  95%% CrI [%.4f, %.4f]\n",
    ate_comb$game[i],
    ate_comb$estimate[i],
    ate_comb$lower[i],
    ate_comb$upper[i]
  ))
}
```

## The posterior linear projections {#sec-proj}

The posterior projection of τ(x) onto the moderator design: a linear summary of which moderators
shift the treatment effect, the Bayesian counterpart to GRF's best linear projection
(@woody2021model).
This is an appendix/robustness summary, not a headline. Terms are ordered by their mean posterior
coefficient across the two games.

quarto-executable-code-5450563D

```r
#| label: fig-bcf-combined-proj
#| fig-cap: "BCF posterior linear projection of the CATE onto the moderator set, both games overlaid, with 95% credible intervals (dodged by game). Each coefficient is the posterior of (X'X)^{-1} X' tau_d. Terms are ordered by their mean coefficient across the two games. Appendix-style linear summary of effect modification."
#| dpi: 500
#| fig-width: 6.5
#| fig-height: 9
proj_ord <- proj_comb |>
  summarise(m = mean(estimate), .by = term) |>
  arrange(m) |>
  pull(term)
proj_plot <- proj_comb |> mutate(term = factor(term, levels = proj_ord))

ggplot(proj_plot, aes(x = estimate, y = term, colour = game, shape = game)) +
  geom_vline(xintercept = 0, linetype = "dashed", colour = "grey60") +
  geom_pointrange(aes(xmin = lower, xmax = upper), position = dodge) +
  scale_colour_game() +
  scale_shape_game() +
  labs(
    title = "Posterior linear projection of the CATE, both games",
    x = "Projection coefficient",
    y = NULL
  ) +
  theme_mbeu()
```

## The posterior CATEs by moderator {#sec-cate}

The headline figures: posterior median CATE on the y-axis, the moderator on the x-axis, with a
cluster-robust 95% interval at each level/grid cell, both games overlaid. The y-axis is free per
facet; discrete levels comprising less than 1% of a game's sample are dropped because their CATE is
too imprecise to compare. The three identity scales are reversed so higher means more attachment,
the three "EU effect on" items are reversed so higher means a more positive evaluation, and
`q_class` runs working class up to higher class with `dont_know` last.

## Conjoint round-level moderators {#sec-cate-prof}

All five conjoint-round (profile-level) moderators on one canvas, Dictator and Trust overlaid.
All five are categorical (including `cj_age`, whose five values are fixed conjoint ages), so each is
drawn as dodged points with cluster-robust intervals.

quarto-executable-code-5450563D

```r
#| label: fig-bcf-combined-prof
#| fig-cap: "BCF posterior estimates of the profile-level CATE ($\\hat{\\tau}_{ric}$) of the Muslim profile across the levels of every conjoint-round moderator (x-axis), Dictator and Trust games overlaid and distinguished jointly by colour, shape, and linetype. The y-axis is $\\hat{\\tau}_{ric}$ on the token-allocation scale. Each point is a per-level posterior median with a cluster-robust 95% interval, formed as the median plus or minus 1.96 times a combined standard error (a normal approximation) that adds the posterior SD of the within-level mean to a respondent cluster-bootstrap SD. Grey bars along the bottom of each facet show the per-level row count (pooled over games). The y-axis is free per facet. In the Profile partisanship facet, partisanship not shown refers to within the co-national category only, since partisanship is displayed for co-national profiles only."
#| dpi: 500
#| fig-width: 9
#| fig-height: 6
plot_comb(
  comb_prof,
  ncol = 3,
  title = bquote(
    "CATE by conjoint-round moderator, both games" ~ (hat(tau)[ric])
  ),
  ylab = yl_ric
)
```

## Individual-level moderators {#sec-cate-resp}

The respondent-level moderators on two figures so the facets stay legible.
Each figure stacks (via `patchwork`) a faceted panel of half the discrete moderators on top with one
of the two genuinely continuous moderators (`q_age`, `q_edu_age_stop`) on a continuous x-axis below,
so the continuous covariate keeps its real histogram and trimmed axis while still sitting beside the
discrete moderators it belongs with.
Ordered numeric scales (the 5- and 7-point items) carry an overlaid cluster-robust 95% ribbon
plus connecting line per game; categorical moderators keep dodged points and intervals.
The three identity scales are reversed so higher means more attachment, the three "EU effect on"
items are reversed so higher means a more positive evaluation, and `q_class` runs working class up
to higher class with `dont_know` last.
The shared game legend is drawn once, from the top panel.

quarto-executable-code-5450563D

```r
#| label: resp-split
resp_cat <- setdiff(resp_mods, c("q_age", "q_edu_age_stop"))
# categorical moderators first, then the ordered scales
resp_cat <- c(intersect(cat_mods, resp_cat), setdiff(resp_cat, cat_mods))
half <- ceiling(length(resp_cat) / 2)
resp_half1 <- resp_cat[1:half]
resp_half2 <- resp_cat[(half + 1):length(resp_cat)]
```

quarto-executable-code-5450563D

```r
#| label: fig-bcf-combined-resp-1
#| fig-cap: "BCF posterior estimates of the respondent-level CATE ($\\hat{\\tau}_i$) of the Muslim profile across the levels of the first half of the discrete individual-level moderators (top panel) and across age (q_age, bottom panel), Dictator and Trust games overlaid and distinguished jointly by colour, shape, and linetype. The y-axis is $\\hat{\\tau}_i$ on the token-allocation scale. All item scales are oriented so that higher x-axis values mean more agreement, a more positive evaluation, or more attachment, depending on the item (see @tbl-wordings). Every point and ribbon value is a posterior median with a cluster-robust 95% interval, formed as the median plus or minus 1.96 times a combined standard error (a normal approximation) that adds the posterior SD to a respondent cluster-bootstrap SD. Top: ordered numeric scales are drawn as a connecting line and interval ribbon per game, categorical moderators as dodged points and intervals. Grey bars show the per-level row count (pooled over games). Bottom: per-grid-cell estimate, interval ribbon, and connecting line per game over a grey histogram of the covariate, on 100 equal intervals across the full 18-75 years. The y-axis is free per facet. Discrete levels comprising less than 1% of a game's sample are dropped because their CATE is too imprecise to compare."
#| dpi: 500
#| fig-width: 11
#| fig-height: 14
p_resp1 <- plot_comb(
  comb_resp |>
    filter(mod %in% resp_half1) |>
    mutate(mod = factor(mod, levels = resp_half1)),
  ncol = 4,
  title = bquote(
    "CATE by individual-level moderator, both games (1 of 2)" ~ (hat(tau)[ic])
  )
)
p_age <- cont_panel(
  comb_resp |> filter(mod == "q_age"),
  m_dict[["q_age"]],
  "q_age",
  c(18, 75)
) +
  guides(colour = "none", shape = "none", fill = "none", linetype = "none")
(p_resp1 / p_age) + plot_layout(heights = c(3, 1))
```

quarto-executable-code-5450563D

```r
#| label: fig-bcf-combined-resp-2
#| fig-cap: "BCF posterior estimates of the respondent-level CATE ($\\hat{\\tau}_i$) of the Muslim profile across the levels of the second half of the discrete individual-level moderators (top panel) and across age at which education stopped (q_edu_age_stop, bottom panel), Dictator and Trust games overlaid and distinguished jointly by colour, shape, and linetype. The y-axis is $\\hat{\\tau}_i$ on the token-allocation scale. All item scales are oriented so that higher x-axis values mean more agreement, a more positive evaluation, or more attachment, depending on the item (see @tbl-wordings). Every ribbon value is a posterior median with a cluster-robust 95% interval, formed as the median plus or minus 1.96 times a combined standard error (a normal approximation) that adds the posterior SD to a respondent cluster-bootstrap SD. Top: ordered numeric scales are drawn as a connecting line and interval ribbon per game. Grey bars show the per-level row count (pooled over games). Bottom: per-year posterior median, interval ribbon, and connecting line per game at each integer year of education-stopping age from 15 to 30, over a grey histogram of the covariate. The y-axis is free per facet. Discrete levels comprising less than 1% of a game's sample are dropped because their CATE is too imprecise to compare."
#| dpi: 500
#| fig-width: 11
#| fig-height: 14
p_resp2 <- plot_comb(
  comb_resp |>
    filter(mod %in% resp_half2) |>
    mutate(mod = factor(mod, levels = resp_half2)),
  ncol = 4,
  title = bquote(
    "CATE by individual-level moderator, both games (2 of 2)" ~ (hat(tau)[ic])
  )
)
p_edu <- cont_panel(
  comb_resp |> filter(mod == "q_edu_age_stop"),
  m_dict[["q_edu_age_stop"]],
  "q_edu_age_stop",
  c(15, 30)
) +
  guides(colour = "none", shape = "none", fill = "none", linetype = "none")
(p_resp2 / p_edu) + plot_layout(heights = c(3, 1))
```

## Session Info {#sec-session-info .appendix .unnumbered}

quarto-executable-code-5450563D

```r
#| label: session-info
#| eval: true
#| echo: true
session_info()
```

## Execution Time {#sec-exec-time .appendix .unnumbered}

quarto-executable-code-5450563D

```r
#| label: exec-time
#| eval: true
#| echo: true
#| include: true
end_time <- Sys.time()
exec_time <- end_time - start_time
cat(paste(
  "R code execution time:",
  round(as.numeric(exec_time, units = "secs"), 2),
  "seconds."
))
```
````

Woody, Spencer, Carlos M. Carvalho, and Jared S. Murray. 2021. “Model Interpretation Through Lower-Dimensional Posterior Summarization.” *Journal of Computational and Graphical Statistics* 30 (1): 144–61. <https://doi.org/10.1080/10618600.2020.1796684>.